# Lot D — Silver : verdict par LLM contraint aux preuves (EXP-D-009)

**Auteur : Membre 5 (Hadil).** Compare, sur les mêmes affirmations :
- le **gabarit + NLI** du Bronze (baseline) ;
- le **LLM** `Qwen/Qwen2.5-7B-Instruct` (4 bits) avec le prompt contrôlé v1 : JSON imposé, citations limitées aux passages fournis, abstention possible, **3 graines** ;
- 4 ablations (indices NLI dans le prompt, 3 puis 1 passage, recherche BM25).

Recherche : hybride BM25 + e5-base (retenue dans EXP-D-008). **Ordre : dev d'abord, test une seule fois.**

Avant de commencer : *Exécution → Modifier le type d'exécution → GPU T4*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Récupérer le projet (branche `lot-d-silver`)
Dépôt privé : créer un token GitHub *classic* (case `repo`, 7 jours), le coller dans la case masquée quand la cellule le demande, puis **le supprimer sur GitHub après usage**. Ne jamais écrire le token dans une cellule.

In [ ]:
import re, subprocess
from getpass import getpass

token = getpass("Colle ton token GitHub puis appuie sur Entrée : ").strip()
url = f"https://{token}@github.com/SouissiOumaima/NewsCheckTN.git"
r = subprocess.run(["git", "clone", "-q", "-b", "lot-d-silver", url], capture_output=True, text=True)
del token, url
print("✅ Projet téléchargé" if r.returncode == 0
      else "❌ " + re.sub(r"gh[pousr]_\w+", "ghp_***", r.stderr))
%cd NewsCheckTN
!git remote set-url origin https://github.com/SouissiOumaima/NewsCheckTN.git
!git log --oneline -1

In [ ]:
!pip install -q -r requirements-dev.txt sentence-transformers bitsandbytes accelerate

## 2. Corpus de test (privé, reconstruit dans la session, jamais poussé sur GitHub)

In [ ]:
!python -m rag_llm.build_tunifact_eval
!python -m rag_llm.freeze_test --check

## 3. Essai rapide (3 affirmations du dev) : vérifier que le modèle se charge et que les sorties sont valides
Le premier lancement télécharge le LLM (~15 Go, 5 à 10 min).

In [ ]:
!python -X utf8 -m rag_llm.silver_verdict --splits dev --limit 3

## 4. Dev complet (20 affirmations)
Regarder surtout la colonne `valid.` (part de sorties JSON valides) et les raisons de rejet.
**Envoyer ce résultat à Claude avant de lancer le test.** Si le prompt doit changer, c'est maintenant (sur le dev), jamais après le test.

In [ ]:
!python -X utf8 -m rag_llm.silver_verdict --splits dev

## 5. Test gelé — une seule fois

In [ ]:
!python -X utf8 -m rag_llm.silver_verdict --splits test

## 6. Récupérer les résultats
Le zip contient les scores, les sorties du LLM et le cache (avec des extraits des passages Tunifact : **privé**, à garder sur ton PC, pas sur GitHub).

In [ ]:
!cd evaluation/results && zip -q ../../lot-d_silver_verdict.zip lot-d_silver_verdict_*.json lot-d_llm_cache.jsonl
from google.colab import files
files.download("lot-d_silver_verdict.zip")